# Orquestador Bronze — Ingesta completa

Ejecuta los 4 notebooks de ingesta de Bronze en secuencia.  
Cada notebook corre en su propio contexto aislado (`dbutils.notebook.run`).

| # | Notebook | Fuente | Destino |
|---|---|---|---|
| 1 | `s3-ine-ingestion-script` | AWS S3 | `ine_*` (5 tablas) |
| 2 | `dropbox-mspas-ingestion-script` | Dropbox API | `dbx_*` (3 tablas) |
| 3 | `localvolume-oms-ingestion-script` | Databricks Volume | `who_guatemala` |
| 4 | `gdrive-oms-ingestion-script` | Google Drive API | `who_costa_rica` |

> **Timeout por notebook:** 3600 s (1 hora). Ajusta `TIMEOUT_SEC` si alguna fuente es más lenta.

In [0]:
import time

# ── Detecta la raíz del repo dinámicamente ────────────────────────────────────
# Funciona para cualquier miembro del equipo sin cambiar rutas hardcodeadas.
_ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
_this_path = _ctx.notebookPath().get()
# _this_path = "/Repos/user@email.com/ss2-dw-project-covid-research/orchestration/run_bronze"
BASE = "/".join(_this_path.split("/")[:-2])  # sube 2 niveles → raíz del repo

TIMEOUT_SEC = 3600  # 1 hora por notebook

_NOTEBOOKS = [
    ("INE — AWS S3",            f"{BASE}/extraction-scripts/s3-ine-ingestion-script"),
    ("MSPAS — Dropbox",         f"{BASE}/extraction-scripts/dropbox-mspas-ingestion-script"),
    ("WHO Guatemala — Volume",   f"{BASE}/extraction-scripts/localvolume-oms-ingestion-script"),
    ("WHO Costa Rica — Drive",   f"{BASE}/extraction-scripts/gdrive-oms-ingestion-script"),
]

print(f"Raíz del repo detectada: {BASE}")
print(f"Notebooks a ejecutar: {len(_NOTEBOOKS)}")

In [0]:
results = []

for label, path in _NOTEBOOKS:
    print(f"\n{'='*60}")
    print(f"▶ Iniciando: {label}")
    print(f"  Path: {path}")
    t0 = time.time()
    try:
        output = dbutils.notebook.run(path, TIMEOUT_SEC)
        elapsed = time.time() - t0
        status = "OK"
        print(f"✓ Completado en {elapsed:.0f}s")
        if output:
            print(f"  Salida: {output}")
    except Exception as e:
        elapsed = time.time() - t0
        status = f"ERROR: {e}"
        print(f"✗ Error en {elapsed:.0f}s: {e}")
    results.append((label, status, f"{elapsed:.0f}s"))

# ── Resumen final ──────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print("RESUMEN BRONZE")
print(f"{'='*60}")
all_ok = True
for label, status, elapsed in results:
    icon = "✓" if status == "OK" else "✗"
    print(f"  {icon} {label:<30} {elapsed:>6}  {status}")
    if status != "OK":
        all_ok = False

print(f"\nEstado final: {'TODOS OK' if all_ok else 'HAY ERRORES — revisa los notebooks individuales'}")
dbutils.notebook.exit("OK" if all_ok else "ERRORS")